# Hunt 2: from one suspicious host to the whole intrusion

Start from a host, pivot to what it touched, then read the case timeline in event-time order.

In [ ]:
# Setup: run from the repository root or the notebooks/ folder. Lab mode is required
# because the cells generate synthetic attack telemetry (nothing is executed on a host).
import os
import sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)
os.environ.setdefault("CYBERPULSE_LAB_MODE", "true")

from datetime import datetime, timezone
from simulator.attack_chain import run_chain
from simulator.benign_baseline import generate_baseline
from soar import views  # noqa: F401  (used by the hunt cells)
from soar.soar_engine import SOAROrchestrator

orchestrator = SOAROrchestrator(policy_mode="DRY_RUN")
for event in generate_baseline(600, 1337):          # seeded benign traffic
    orchestrator.process_event(event)
run_chain(orchestrator, start=datetime(2026, 11, 1, 9, 0, tzinfo=timezone.utc))  # ten-stage synthetic intrusion
print(len(orchestrator.event_store.snapshot()), "events in the hunting store")

In [ ]:
p = views.pivot(orchestrator, "host", "CORP-FINANCE-02")
print(p["events"], "events,", p["alerted"], "alerted")
for field, items in p["related"].items():
    print(f"{field:>15}: " + ", ".join(f'{i["value"]} ({i["count"]})' for i in items[:4]))

In [ ]:
case_id = orchestrator.case_manager.find_by_incident(orchestrator.dfir_engine.incident_list[-1]["incident_id"])["case_id"]
timeline = views.case_timeline(orchestrator, case_id)
for e in timeline["entries"]:
    print(e["at"][:19], e["tactic"], e["technique"], e["rule"])